# 탐색적 데이터 분석 — 질문에서 새 질문까지


**설명 전용 자료:** 개념·예제·주의점을 Markdown 셀로 제공합니다. 연습문제와 빈 풀이 셀은 같은 이름의 `_Practice.ipynb`에 있습니다.

**학습 목표:** EDA의 목적을 설명하고 질문 → 탐색 → 인사이트 → 새 질문의 반복 과정을 데이터 품질 점검과 함께 수행합니다.

## 처음 배우는 분을 위한 안내

EDA는 계산이나 그림으로 자료를 살펴 질문을 구체화하는 과정입니다. 데이터 사전은 열 이름과 뜻을 정리한 표입니다. order_id=주문 번호, customer_id=고객 번호, revenue=주문 금액, quantity=수량, unit_price=단가, region=지역, delivery_days=배송일수, customer_rating=평점입니다. customer의 churn은 Churn(이탈)·Stay(유지), usage_count는 이용 횟수입니다.

**학습 순서:** 문제 1~5는 필수 기초, 6~12는 선택 도전입니다. orders_raw는 보존본, orders는 실습 복사본입니다. 동일 주문 ID가 있어 처음부터 한 행을 고유 주문이라고 확정하지 않습니다.

셀은 실행 단위입니다. Shift+Enter로 한 셀씩 위에서 아래로 실행합니다. 화면에 보이는 ```python``` 회색 코드 블록은 설명 문서이므로 실행하려면 새 코드 셀에 복사하세요. 준비 셀은 데이터만 만들며 문제의 답은 포함하지 않습니다. 빈 풀이 셀은 오류 없이 실행되지만 아직 결과를 계산한 것은 아닙니다. **완료할 때는 계산 결과 하나와 그 의미 한 문장을 함께 남기세요.**

## 예제 데이터

아래 가상 데이터를 공통으로 사용합니다. 코드 블록은 읽기용이며 실행하려면 별도 코드 셀에 복사하세요. 예제는 순서대로 이어집니다. 독립 실행이 필요한 경우 이 준비 코드를 먼저 실행하세요.

```python
import pandas as pd

# 의도적으로 결측값·극단값·동일 ID가 있는 학습용 가상 데이터입니다.
# 실제 고객·원본 강의 데이터의 분석 결과로 일반화하지 마세요.
orders_raw = pd.DataFrame({
    'order_id': [101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 110],
    'customer_id': ['A', 'B', 'A', 'C', 'D', 'B', 'E', 'F', 'G', 'H', 'H'],
    'order_date': ['2024-01-01', '2024-01-02', '2024-01-03', '2024-01-04',
                   '2024-01-05', '2024-02-01', '2024-02-02', '2024-02-03',
                   '2024-02-04', '2024-02-05', '2024-02-05'],
    'product_category': ['Food', 'Book', 'Food', 'Tech', 'Book', 'Tech',
                         'Food', 'Book', 'Tech', 'Food', 'Food'],
    'region': ['South', 'North', 'South', 'East', None, 'South',
               'East', 'North', 'South', 'East', 'East'],
    'quantity': [2, 1, 3, 1, 2, 2, 1, 2, 1, 2, 2],
    'unit_price': [100, 200, 100, 1000, 200, 1000, 100, 200, 1000, 100, 100],
    'revenue': [200, 200, 300, 1000, None, 2000, 100, 400, 50000, 200, 200],
    'delivery_days': [1, 3, 2, 8, 4, 7, 9, 3, 12, 2, 2],
    'customer_rating': [4.5, 4.0, 4.8, 2.0, 4.1, 3.0, 2.5, 4.3, 1.0, 4.6, 4.6]
})
orders_raw['order_date'] = pd.to_datetime(orders_raw['order_date'])
orders = orders_raw.copy(deep=True)
customers = pd.DataFrame({
    'customer': list('ABCDEF'), 'usage_count': [3, 4, 2, 18, 15, 20],
    'churn': ['Churn', 'Churn', 'Churn', 'Stay', 'Stay', 'Stay']
})
```

## EDA는 무엇을 하는가

EDA는 데이터의 구조·분포·품질·변수 관계를 살펴 분석 질문을 구체화하는 과정입니다. 통계표나 그래프를 많이 만드는 것 자체가 목표는 아닙니다. 무엇을 알고 싶은지 정하고, 그 질문에 필요한 관측 단위·기간·분모를 확인합니다.

이 자료에서는 쇼핑몰·고객 이탈의 작은 표로 결측·중복·범위와 변수 관계를 차례로 점검합니다. 제공 데이터는 가상이므로 발견한 패턴을 실제 사업이나 고객 행동으로 일반화하지 않습니다.

## EDA를 하는 세 가지 이유

| 목적 | 확인 질문 | 활용 |
| --- | --- | --- |
| 데이터 이해 | 한 행·열·기간은 무엇인가? | 분석 단위와 범위 결정 |
| 품질 문제 발견 | 누락·중복·불가능한 값이 있는가? | 추가 확인과 전처리 방향 |
| 변수 선택의 단서 | 그룹 간 차이나 관련성이 있는가? | 다음 탐색·모델링 질문 |

데이터를 받은 직후 발견한 극단값은 오류일 수도 실제 대량 주문일 수도 있습니다. 자동 삭제가 아니라 원인 확인 대상으로 남깁니다. EDA와 전처리는 연결되지만 '문제 발견'과 '값 수정'을 구분해 기록합니다.

## 질문 → 탐색 → 인사이트 → 새 질문

예: '어떤 카테고리의 매출이 높은가?' → 그룹별 합계·행 수·중앙값 탐색 → '가상 자료의 Tech 합계가 크고 한 행의 영향도 크다' → '단가 때문인가, 주문 수 때문인가, 데이터 오류인가?'처럼 구체화합니다.

인사이트는 실제 수치와 비교 기준을 포함해야 합니다. 'Tech가 좋다'보다는 '누락값을 제외한 매출 합계에서 Tech가 가장 크다. 단일 극단값의 영향을 추가 확인해야 한다'가 검증 가능한 표현입니다.

```python
category = orders.groupby('product_category').agg(
    total_revenue=('revenue', 'sum'),
    valid_revenue=('revenue', 'count'),
    rows=('order_id', 'size'),
    median_revenue=('revenue', 'median'))
print(category.sort_values('total_revenue', ascending=False))
```

## 기본 구조·누락·중복 점검

shape와 info로 구조를, isna로 결측을 확인합니다. 결측률의 분모는 검사 대상 행 수입니다. 완전히 같은 행인지, 주문 ID가 같지만 정보가 다른 행인지도 나눠 봅니다. 동일 ID를 발견했다고 바로 삭제하면 실제 복수 항목 주문을 잃을 수 있으므로 업무 정의가 필요합니다.

```python
print(orders.shape)  # (11, 10)
orders.info()
quality = pd.DataFrame({'missing_count': orders.isna().sum(),
                        'missing_rate': orders.isna().mean()})
print(quality)
print(orders.duplicated().sum())  # 1
print(orders[orders.duplicated('order_id', keep=False)])
```

## 분포와 극단값의 영향

describe에서 count·평균·중앙값·최댓값을 같이 봅니다. 작은 표본에서 한 값은 평균과 합계를 크게 움직입니다. IQR 규칙은 추가 점검 후보를 찾는 도구이며 오류를 증명하지 않습니다. 결측을 0으로 채우면 '관측 안 됨'이 '매출 0'으로 바뀌므로 의미를 먼저 확인해야 합니다.

```python
print(orders['revenue'].describe())
q1 = orders['revenue'].quantile(0.25)
q3 = orders['revenue'].quantile(0.75)
iqr = q3 - q1
candidates = orders.loc[(orders['revenue'] < q1 - 1.5 * iqr) |
                        (orders['revenue'] > q3 + 1.5 * iqr)]
print(candidates[['order_id', 'revenue']])
```

## 이탈 고객 비교와 인과관계

강의 예제처럼 이용 횟수를 이탈 여부별로 비교하면 관련성의 단서를 얻을 수 있습니다. 하지만 이용이 적어서 이탈했는지, 이탈이 가까워져 이용이 줄었는지, 다른 요인인지 구분할 수 없습니다. 그룹 표본 수·중앙값·관측 기간을 함께 적습니다. 이탈 이후 정보를 예측 변수로 쓰면 미래 정보 누출이 될 수 있습니다.

```python
usage = customers.groupby('churn').agg(
    customer_count=('customer', 'size'),
    mean_usage=('usage_count', 'mean'),
    median_usage=('usage_count', 'median'))
print(usage)  # Churn 평균 3, Stay 평균 약 17.67
```

## 시각화는 질문에 맞춰 선택

히스토그램은 단일 수치의 분포, 막대그래프는 그룹별 집계, 산점도는 두 수치의 관계를 봅니다. 축·단위·표본 수·결측 처리 조건을 적습니다. 극단값 때문에 나머지 값이 좁게 보이면 축 범위나 별도 비교를 검토하되 제외 조건을 숨기지 않습니다. 다음은 추가 실습 예제이며 matplotlib이 필요합니다.

```python
import matplotlib.pyplot as plt

orders['revenue'].dropna().plot.hist(bins=8)
plt.xlabel('Revenue (synthetic units)')
plt.title('Observed revenue distribution')
plt.tight_layout()
plt.show()

orders.plot.scatter(x='delivery_days', y='customer_rating')
plt.title('Delivery days vs rating (synthetic sample)')
plt.tight_layout()
plt.show()
```

## 민감도 비교와 보고서

원본을 보존하고 비교 조건만 달리해 결과가 얼마나 변하는지 봅니다. '원본 전체', '완전 중복 제외', '특정 극단값 임시 제외'는 서로 다른 분석 시나리오입니다. 임시 제외 결과를 정제된 최종 정답으로 부르지 마세요.

좋은 짧은 보고서는 질문·데이터 범위·처리 조건·수치 근거·한계·다음 질문을 포함합니다. 작은 가상 표본의 상관계수나 그룹 차이를 실제 인과 효과처럼 쓰지 않습니다. 누락된 값은 누락 사실과 해당 분모를 남깁니다.

```python
scenarios = {
    'raw': orders,
    'exact_duplicates_removed': orders.drop_duplicates(),
    'temporary_high_value_excluded': orders.loc[orders['revenue'].isna() |
                                               (orders['revenue'] <= 10000)]
}
for label, frame in scenarios.items():
    print(label, len(frame), frame['revenue'].count(),
          frame['revenue'].mean(), frame['revenue'].median())
```

## 복습 순서

1. 개념과 입력·출력의 분석 단위를 확인합니다.
2. 작은 예제에서 결과를 손으로 예상합니다.
3. 연습용 노트북의 준비 셀을 실행하고 문제를 풉니다.
4. 결과 수치와 점검 기준을 비교하고 이유를 기록합니다.
5. 기본 문제를 마친 뒤 선택 도전을 진행합니다. 입력을 바꾸면 결과도 달라지는 이유를 설명합니다.

## 선택 도전: 작은 표에서 누락값과 큰 값 구분하기

**필수 아님:** 위의 기본 문제를 먼저 완료하세요. 새 파일을 구하거나 준비 셀의 변수를 교체하지 않습니다. 아래 작은 입력만 별도 코드 셀에 복사해 사용합니다.

### 입력

```python
import pandas as pd
extra_sales = pd.Series([100,120,None,5000],name='revenue')
```

### 진행 순서

1. 전체 길이와 count()로 센 비결측 수를 각각 출력합니다.
2. 평균과 중앙값을 구합니다.
3. 5000이 실제 큰 주문인지 입력 오류인지 확인할 질문을 한 개 적습니다. 값을 삭제하지 않습니다.

**완료 기준:** 전체 4개, 비결측 3개, 평균 1740, 중앙값 120. 큰 값만으로 오류라고 확정하지 않음.

### 선택 도전 풀이

```python
print(len(extra_sales),extra_sales.count(),extra_sales.mean(),extra_sales.median())
assert len(extra_sales)==4 and extra_sales.count()==3
assert extra_sales.mean()==1740 and extra_sales.median()==120
```